# 00 Setup

Se ejecuta una vez, a mano, antes del job. Crea el volume, las carpetas, las tablas
y los permisos del grupo `mine-group8`. Carga el World Port Index en `oceanwatch_puertos`.

Catálogo `dbr_acad_ingsistemas_mine`, esquema `default`. En el cluster los archivos van con Python sobre `/Volumes`.


In [ ]:
%run ./_config


## Volume y carpetas

`raw_zips/` guarda el zip, `landing/` el CSV que lee bronze, `checkpoints/` los
checkpoints de los streams, `ref/` el World Port Index y `_schemas/` el esquema de Auto Loader.


In [ ]:
import os

spark.sql(f"""
CREATE VOLUME IF NOT EXISTS {VOLUME_NAME}
COMMENT 'Archivos de OceanWatch: zips NOAA, CSV de landing, checkpoints y referencias.'
""")

for ruta in (RAW_ZIPS, LANDING, CHECKPOINTS, SCHEMAS, REF):
    os.makedirs(ruta, exist_ok=True)
    print("Listo", ruta)


## Tablas

Bronze guarda los datos crudos, tal como llegaron. Silver guarda una fila por
`(MMSI, BaseDateTime)` y `fecha_dia`, con `CLUSTER BY (fecha_dia)`. Cuarentena
guarda las filas que no pasan las reglas, con `motivo`. `oceanwatch_metricas`
anota cada ejecución de bronze y cada batch de silver. Las tres tablas gold se rehacen por `fecha`.


In [ ]:
tablas = {
    BRONZE: """
        (
          MMSI STRING,
          BaseDateTime TIMESTAMP,
          LAT DOUBLE,
          LON DOUBLE,
          SOG DOUBLE,
          COG DOUBLE,
          Heading DOUBLE,
          VesselName STRING,
          IMO STRING,
          CallSign STRING,
          VesselType INT,
          Status INT,
          Length DOUBLE,
          Width DOUBLE,
          Draft DOUBLE,
          Cargo STRING,
          TransceiverClass STRING,
          _source_file STRING COMMENT 'Ruta del CSV en landing.',
          _ingested_at TIMESTAMP COMMENT 'Instante en que el batch escribio la fila.'
        )
        USING DELTA
        COMMENT 'Bronze AIS: mensaje crudo, tal como llego. Cada archivo se lee una vez. La deduplicacion es en silver.'
    """,
    SILVER: """
        (
          MMSI STRING,
          BaseDateTime TIMESTAMP,
          LAT DOUBLE,
          LON DOUBLE,
          SOG DOUBLE,
          COG DOUBLE,
          Heading DOUBLE,
          VesselName STRING,
          IMO STRING,
          CallSign STRING,
          VesselType INT,
          Status INT,
          Length DOUBLE,
          Width DOUBLE,
          Draft DOUBLE,
          Cargo STRING,
          TransceiverClass STRING,
          _source_file STRING,
          _ingested_at TIMESTAMP,
          fecha_dia DATE
        )
        USING DELTA
        CLUSTER BY (fecha_dia)
        COMMENT 'Silver AIS: una fila por (MMSI, BaseDateTime) en cada fecha_dia. La deduplicacion es aqui.'
    """,
    CUARENTENA: """
        (
          MMSI STRING,
          BaseDateTime TIMESTAMP,
          LAT DOUBLE,
          LON DOUBLE,
          SOG DOUBLE,
          COG DOUBLE,
          Heading DOUBLE,
          VesselName STRING,
          IMO STRING,
          CallSign STRING,
          VesselType INT,
          Status INT,
          Length DOUBLE,
          Width DOUBLE,
          Draft DOUBLE,
          Cargo STRING,
          TransceiverClass STRING,
          _source_file STRING,
          _ingested_at TIMESTAMP,
          motivo STRING COMMENT 'Reglas fallidas, separadas por coma.'
        )
        USING DELTA
        COMMENT 'Filas AIS que no entran a silver, con el motivo.'
    """,
    METRICAS: """
        (
          fecha DATE COMMENT 'Dia del parametro del job.',
          etapa STRING COMMENT 'bronze o silver.',
          filas_antes BIGINT COMMENT 'Conteo de la tabla bronze antes del stream.',
          filas_despues BIGINT COMMENT 'Conteo de la tabla bronze despues del stream.',
          filas_entrada BIGINT COMMENT 'Filas del batch de silver.',
          filas_validas BIGINT COMMENT 'Filas del batch que pasan las reglas.',
          filas_cuarentena BIGINT COMMENT 'Filas del batch que van a cuarentena.',
          ejecutado_en TIMESTAMP
        )
        USING DELTA
        COMMENT 'Conteos de cada ejecucion de bronze y de cada batch de silver.'
    """,
    PUERTOS: """
        (
          numero INT COMMENT 'World Port Index Number.',
          nombre STRING,
          pais STRING,
          latitud DOUBLE,
          longitud DOUBLE
        )
        USING DELTA
        COMMENT 'World Port Index (UpdatedPub150), el mismo catalogo de la Parte 1.'
    """,
    GOLD_TRAY: """
        (
          fecha DATE,
          MMSI STRING,
          n_posiciones BIGINT,
          inicio TIMESTAMP,
          fin TIMESTAMP,
          lat_inicio DOUBLE,
          lon_inicio DOUBLE,
          lat_fin DOUBLE,
          lon_fin DOUBLE,
          distancia_km DOUBLE COMMENT 'Suma de tramos haversine del dia, radio 6371 km.',
          sog_media DOUBLE
        )
        USING DELTA
        PARTITIONED BY (fecha)
        COMMENT 'Una trayectoria por buque y dia, calculada sobre silver.'
    """,
    GOLD_PUERTO: """
        (
          fecha DATE,
          MMSI STRING,
          puerto STRING,
          pais STRING,
          n_posiciones BIGINT,
          inicio TIMESTAMP,
          fin TIMESTAMP
        )
        USING DELTA
        PARTITIONED BY (fecha)
        COMMENT 'Buque con al menos una posicion del dia junto a un puerto del WPI.'
    """,
    GOLD_TRAFICO: """
        (
          fecha DATE,
          hora INT COMMENT 'Hora UTC de BaseDateTime.',
          h3_cell STRING COMMENT 'Celda H3 resolucion 8, la misma de la Parte 1.',
          n_posiciones BIGINT,
          n_buques BIGINT
        )
        USING DELTA
        PARTITIONED BY (fecha)
        COMMENT 'Matriz de trafico celda H3 por hora, un dia.'
    """,
}

for nombre, cuerpo in tablas.items():
    spark.sql(f"CREATE TABLE IF NOT EXISTS {nombre} {cuerpo}")
    print("Tabla", nombre)



## World Port Index

La Parte 1 usó `UpdatedPub150.csv` (`Main Port Name`, `Country Code`, `Latitude`,
`Longitude`). El número de puerto en el CSV viene como `7950.0`: se lee como
double y se guarda como int. Spark no puede leer el archivo del repo (eso pide
ANY FILE), así que Python lo copia desde la carpeta Git a `ref/` y la lectura
es sobre `/Volumes`.


In [ ]:
from pathlib import Path
import shutil
from pyspark.sql import functions as F

destino = Path(REF) / "UpdatedPub150.csv"
destino.parent.mkdir(parents=True, exist_ok=True)

def csv_en_repo() -> Path | None:
    """CSV en la raiz del repo, al lado de src/, visto desde la carpeta Git."""
    try:
        ruta_nb = (
            dbutils.notebook.entry_point.getDbutils()
            .notebook()
            .getContext()
            .notebookPath()
            .get()
        )
    except Exception:
        return None
    bases = [ruta_nb]
    if not ruta_nb.startswith("/Workspace"):
        bases.append("/Workspace" + ruta_nb)
    # src/00_setup -> raiz del repo. Se prueban las dos formas del path.
    for base in bases:
        candidato = Path(base).parent.parent / "UpdatedPub150.csv"
        if candidato.is_file():
            return candidato
    return None

origen = csv_en_repo()
if origen is not None:
    shutil.copyfile(origen, destino)
    print("Copiado", origen, "->", destino)
elif not destino.is_file():
    raise FileNotFoundError(
        "No esta UpdatedPub150.csv en el repo ni en "
        f"{destino}. Abre el notebook desde la carpeta Git del repo."
    )
else:
    print("Se usa la copia ya presente en", destino)

# El numero viene como '7950.0'. String -> int falla. Double y luego int.
(
    spark.read.option("header", True).csv(str(destino))
    .select(
        F.expr("try_cast(`World Port Index Number` AS DOUBLE)").cast("int").alias("numero"),
        F.col("Main Port Name").alias("nombre"),
        F.col("Country Code").alias("pais"),
        F.expr("try_cast(Latitude AS DOUBLE)").alias("latitud"),
        F.expr("try_cast(Longitude AS DOUBLE)").alias("longitud"),
    )
    .where(F.col("latitud").isNotNull() & F.col("longitud").isNotNull() & F.col("numero").isNotNull())
    .write.mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(PUERTOS)
)
spark.sql(
    f"COMMENT ON TABLE {PUERTOS} IS "
    "'World Port Index (UpdatedPub150), el mismo catalogo de la Parte 1.'"
)
print("Puertos", spark.table(PUERTOS).count())


## Permisos

El grupo `mine-group8` puede leer y escribir el volume, y leer y modificar las tablas.


In [ ]:
group = "`mine-group8`"
harvy = "`h.benitez@uniandes.edu.co`"
luis = "`l.galvan@uniandes.edu.co`"

spark.sql(f"GRANT READ VOLUME ON VOLUME {VOLUME_NAME} TO {group}")
spark.sql(f"GRANT WRITE VOLUME ON VOLUME {VOLUME_NAME} TO {group}")

spark.sql(f"GRANT READ VOLUME ON VOLUME {VOLUME_NAME} TO {harvy}")
spark.sql(f"GRANT WRITE VOLUME ON VOLUME {VOLUME_NAME} TO {harvy}")

spark.sql(f"GRANT READ VOLUME ON VOLUME {VOLUME_NAME} TO {luis}")
spark.sql(f"GRANT WRITE VOLUME ON VOLUME {VOLUME_NAME} TO {luis}")

for tabla in (BRONZE, SILVER, CUARENTENA, METRICAS, PUERTOS, GOLD_TRAY, GOLD_PUERTO, GOLD_TRAFICO):
    spark.sql(f"GRANT SELECT ON TABLE {tabla} TO {group}")
    spark.sql(f"GRANT MODIFY ON TABLE {tabla} TO {group}")
    spark.sql(f"GRANT SELECT ON TABLE {tabla} TO {harvy}")
    spark.sql(f"GRANT MODIFY ON TABLE {tabla} TO {harvy}")
    spark.sql(f"GRANT SELECT ON TABLE {tabla} TO {luis}")
    spark.sql(f"GRANT MODIFY ON TABLE {tabla} TO {luis}")
    print("GRANT", tabla)

## Reiniciar todo (opcional)

Esta celda no hace nada mientras `REINICIAR_TODO` sea `False`.

Con `True` borra todas las tablas `oceanwatch_` de este setup (bronze, silver,
cuarentena, métricas, puertos y las tres gold), el contenido de `checkpoints/`
(bronze y silver) y los archivos de `landing/`. Los zip de `raw_zips/` se
quedan, así `01_ingesta` no vuelve a bajar NOAA: solo republica el CSV.

Pasos:

1. Ejecuta `%run ./_config`.
2. Pon `REINICIAR_TODO = True` y ejecuta esta celda.
3. Vuelve a dejar `REINICIAR_TODO = False`.
4. Ejecuta otra vez las celdas de volume, tablas, World Port Index y permisos.
5. Ejecuta `01_ingesta`, `02_bronze`, `03_silver` y `04_gold`, en ese orden.


In [ ]:
import shutil
from pathlib import Path

REINICIAR_TODO = False

TABLAS = (
    BRONZE,
    SILVER,
    CUARENTENA,
    METRICAS,
    PUERTOS,
    GOLD_TRAY,
    GOLD_PUERTO,
    GOLD_TRAFICO,
)


def vaciar_directorio(ruta: Path) -> None:
    if not ruta.exists():
        print("No existe", ruta)
        return
    for hijo in ruta.iterdir():
        if hijo.is_dir():
            shutil.rmtree(hijo)
        else:
            hijo.unlink()
        print("Borrado", hijo)


if not REINICIAR_TODO:
    print(
        "Opcional. Pon REINICIAR_TODO = True para borrar las tablas oceanwatch_, "
        "checkpoints y landing. Los zip de raw_zips se conservan."
    )
else:
    for tabla in TABLAS:
        spark.sql(f"DROP TABLE IF EXISTS {tabla}")
        print("DROP", tabla)
    vaciar_directorio(Path(CHECKPOINTS))
    vaciar_directorio(Path(LANDING))
    print("raw_zips se conserva:", RAW_ZIPS)
    print("Listo. Ejecuta volume, tablas, WPI y permisos. Despues 01, 02, 03 y 04.")
